# Notebook 01 — Acquisition & Exploration des Données
**Projet :** Système Intelligent de Détection de Spam et de Phishing  
**Auteur :** Nghogué Taptué Franck Roddier — 5GI, ENSPY  
**Semaine :** 1 / 3

---

## Objectifs de ce notebook
1. Installer et charger les datasets publics (SpamAssassin, Enron, PhishingCorpus)
2. Inspecter la structure brute des emails
3. Analyser la distribution des classes (spam vs ham vs phishing)
4. Visualiser les déséquilibres et les caractéristiques textuelles
5. Produire un fichier `emails_raw.csv` prêt pour le Notebook 02

## 1. Installation des dépendances

In [ ]:
# Installation des bibliothèques nécessaires
# À exécuter une seule fois (commenter ensuite)
!pip install pandas numpy matplotlib seaborn scikit-learn requests tqdm wordcloud

In [ ]:
import os
import re
import email
import tarfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from wordcloud import WordCloud

# Configuration globale des graphiques
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# Dossier de travail
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print('✅ Environnement prêt')

## 2. Chargement des datasets

Nous utilisons trois sources complémentaires :
| Dataset | Contenu | Classes |
|---------|---------|--------|
| **SpamAssassin** | Emails anglais annotés | spam / ham |
| **Enron Spam** | Emails d'entreprise | spam / ham |
| **PhishingCorpus** | Emails de phishing | phishing / ham |

> **Note :** Les URLs de téléchargement pointent vers des sources académiques publiques. Si l'accès réseau est limité, un jeu de données synthétique est généré automatiquement (section 2.4).

In [ ]:
# ============================================================
# 2.1  SpamAssassin Public Corpus
# https://spamassassin.apache.org/old/publiccorpus/
# ============================================================

SA_URLS = {
    'spam': 'https://spamassassin.apache.org/old/publiccorpus/20050311_spam_2.tar.bz2',
    'ham':  'https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham.tar.bz2',
}

def download_and_extract(url: str, dest_dir: Path, label: str) -> list[dict]:
    """
    Télécharge un archive tar.bz2 et extrait le texte brut de chaque email.
    Retourne une liste de dicts {text, label, source}.
    """
    archive_path = dest_dir / f'{label}.tar.bz2'
    
    # Téléchargement (avec barre de progression)
    if not archive_path.exists():
        print(f'⬇  Téléchargement {label}...')
        try:
            response = requests.get(url, stream=True, timeout=30)
            response.raise_for_status()
            total = int(response.headers.get('content-length', 0))
            with open(archive_path, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
                    bar.update(len(chunk))
        except Exception as e:
            print(f'⚠  Téléchargement échoué ({e}). Passage au dataset synthétique.')
            return []
    else:
        print(f'📦 Archive {label} déjà présente.')

    # Extraction
    records = []
    try:
        with tarfile.open(archive_path, 'r:bz2') as tar:
            for member in tqdm(tar.getmembers(), desc=f'Extraction {label}'):
                if member.isfile():
                    f = tar.extractfile(member)
                    if f:
                        raw = f.read()
                        try:
                            text = raw.decode('utf-8', errors='replace')
                        except Exception:
                            text = raw.decode('latin-1', errors='replace')
                        records.append({'text': text, 'label': label, 'source': 'spamassassin'})
    except Exception as e:
        print(f'⚠  Erreur extraction {label}: {e}')
    
    print(f'   → {len(records)} emails extraits ({label})')
    return records


# Téléchargement SpamAssassin
records_sa_spam = download_and_extract(SA_URLS['spam'], DATA_DIR, 'spam')
records_sa_ham  = download_and_extract(SA_URLS['ham'],  DATA_DIR, 'ham')

In [ ]:
# ============================================================
# 2.2  Dataset synthétique de phishing
#      (utilisé si le téléchargement est impossible,
#       ou en complément pour la classe 'phishing')
# ============================================================

PHISHING_SAMPLES = [
    "Dear customer, your account has been suspended. Click here to verify: http://secure-bank-login.xyz/verify?id=1234",
    "URGENT: Confirm your PayPal details immediately or your account will be closed. Login: http://paypa1.com/login",
    "Your password will expire in 24 hours. Update now: http://microsoft-security-alert.com/update",
    "Congratulations! You've been selected for a $1000 gift card. Claim here: http://prize-winner.net/claim",
    "IT Support: Your mailbox is full. Please login to free space: http://webmail-login.co/free",
    "Dear user, suspicious activity detected on your account. Verify now: http://amazon-security.info/verify",
    "Your invoice is ready. Download at: http://invoice-download.xyz/doc?id=9871 - Action required.",
    "Alert: Your DHL package is on hold. Confirm delivery: http://dhl-tracking.xyz/confirm?pkg=5523",
    "Netflix: Your payment failed. Update billing info: http://netflix-billing-update.com/fix",
    "HR Department: Review your updated contract: http://company-hr-portal.info/contract?emp=1124",
    "Security Alert: Sign-in from new device detected. Verify identity: http://google-security.xyz/verify",
    "Your crypto wallet requires verification. Act now: http://coinbase-secure.net/verify",
    "IRS Notice: You are eligible for a tax refund. Claim: http://irs-refund.info/claim?id=4499",
    "Dear client, please review the attached contract and reply within 24h. http://docusign-secure.biz/review",
    "Bank of America: Unusual login attempt. Secure account: http://boa-security-portal.net/secure",
]

HAM_SAMPLES = [
    "Hi team, please find attached the quarterly report for Q3. Let me know if you have questions.",
    "Meeting rescheduled to Friday at 10am. Agenda: product roadmap review.",
    "Thanks for your order #4821. Your package will arrive within 3-5 business days.",
    "Reminder: your dentist appointment is confirmed for tomorrow at 2:30 PM.",
    "Here is the project proposal you requested. Please review by end of week.",
    "Your monthly bank statement is now available in your online banking portal.",
    "Hello, I wanted to follow up on our discussion from last week regarding the budget.",
    "Newsletter: Top 5 Python libraries for data science in 2024.",
    "Your subscription has been renewed. Thank you for staying with us.",
    "Please find the minutes from today's meeting attached to this email.",
    "Congratulations on completing the online course! Your certificate is attached.",
    "Your flight booking is confirmed: Paris → London on 15 June at 07:45.",
    "Invitation: Annual company dinner on December 10th. RSVP by November 30.",
    "The IT department will perform maintenance on Saturday from 2-4 AM.",
    "Your job application has been received. We will contact you within 2 weeks.",
]

SPAM_SAMPLES = [
    "Buy cheap Viagra online! Lowest prices guaranteed. No prescription needed.",
    "Make $5000 a week from home! No experience required. Start today!",
    "You have won a free iPhone 15! Click here to claim your prize NOW!",
    "LIMITED TIME: 90% off luxury watches. Authentic brands at discount prices!",
    "Lose 20 pounds in 2 weeks with this miracle pill. Doctor approved!",
    "CASINO BONUS: $500 free credit. No deposit required. Play now!",
    "Work from home opportunity! Earn unlimited income with our MLM system.",
    "Hot singles in your area want to meet you tonight! Click here!",
    "Free credit score check. No card required. Get yours instantly!",
    "URGENT: You have an unclaimed inheritance of $8.5 million. Contact us.",
    "Best mortgage rates! Refinance now and save thousands. Apply online!",
    "Enlarge your manhood naturally. Proven method. 100% satisfaction guaranteed.",
    "You are pre-approved for a $10,000 loan. Bad credit OK. Apply now!",
    "Download any movie for FREE! Unlimited streaming, no subscription!",
    "Double your Bitcoin investment in 24 hours. Guaranteed returns!",
]

# Construction du dataset synthétique
synthetic_records = []
for text in PHISHING_SAMPLES:
    synthetic_records.append({'text': text, 'label': 'phishing', 'source': 'synthetic'})
for text in HAM_SAMPLES:
    synthetic_records.append({'text': text, 'label': 'ham', 'source': 'synthetic'})
for text in SPAM_SAMPLES:
    synthetic_records.append({'text': text, 'label': 'spam', 'source': 'synthetic'})

print(f'Dataset synthétique : {len(synthetic_records)} emails')

In [ ]:
# ============================================================
# 2.3  Fusion de toutes les sources
# ============================================================

all_records = records_sa_spam + records_sa_ham + synthetic_records

# Si les téléchargements ont échoué, on utilise uniquement le synthétique
if len(all_records) == len(synthetic_records):
    print('⚠  Seul le dataset synthétique est disponible.')
    print('   Pour un projet réel, téléchargez manuellement les corpus SpamAssassin/Enron.')

df = pd.DataFrame(all_records)

# Nettoyage minimal : suppression des emails vides
df = df[df['text'].str.strip().str.len() > 10].reset_index(drop=True)

# Ajout de métadonnées utiles
df['char_count']  = df['text'].str.len()
df['word_count']  = df['text'].str.split().str.len()
df['url_count']   = df['text'].str.count(r'https?://')
df['has_html']    = df['text'].str.contains(r'<[^>]+>', regex=True)
df['exclamation'] = df['text'].str.count('!')

print(f'\n📊 Dataset final : {len(df)} emails')
print(df['label'].value_counts())
df.head()

## 3. Analyse exploratoire (EDA)

In [ ]:
# ============================================================
# 3.1  Distribution des classes
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Camembert ---
class_counts = df['label'].value_counts()
colors = {'ham': '#2ecc71', 'spam': '#e74c3c', 'phishing': '#e67e22'}
pie_colors = [colors.get(l, '#95a5a6') for l in class_counts.index]

axes[0].pie(
    class_counts.values,
    labels=class_counts.index,
    colors=pie_colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Distribution des classes', fontweight='bold')

# --- Barplot avec valeurs ---
bars = axes[1].bar(
    class_counts.index,
    class_counts.values,
    color=pie_colors,
    edgecolor='white',
    linewidth=2
)
axes[1].set_title('Nombre d\'emails par classe', fontweight='bold')
axes[1].set_ylabel('Nombre d\'emails')
for bar, val in zip(bars, class_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                 str(val), ha='center', va='bottom', fontweight='bold')

plt.suptitle('Analyse du déséquilibre des classes', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('data/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Observation : Le déséquilibre de classes est un risque majeur (voir section 1.3 du cahier de charges).')

In [ ]:
# ============================================================
# 3.2  Distribution de la longueur des emails
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, xlabel in zip(
    axes,
    ['word_count', 'char_count'],
    ['Nombre de mots', 'Nombre de caractères']
):
    for label in df['label'].unique():
        subset = df[df['label'] == label][col]
        # Clip pour la lisibilité
        subset = subset.clip(upper=subset.quantile(0.95))
        ax.hist(subset, bins=40, alpha=0.6, label=label, color=colors.get(label, 'gray'))
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Fréquence')
    ax.set_title(f'Distribution — {xlabel}')
    ax.legend()

plt.suptitle('Longueur des emails par classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data/email_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 3.3  Analyse des URLs et marqueurs caractéristiques
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

features = ['url_count', 'exclamation', 'has_html']
titles   = ['Nombre d\'URLs', 'Points d\'exclamation (!)', 'Présence de HTML']

for ax, feat, title in zip(axes, features, titles):
    grouped = df.groupby('label')[feat].mean()
    bar_colors = [colors.get(l, 'gray') for l in grouped.index]
    bars = ax.bar(grouped.index, grouped.values, color=bar_colors, edgecolor='white', linewidth=2)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Valeur moyenne')
    for bar, val in zip(bars, grouped.values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
                f'{val:.2f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Marqueurs caractéristiques par classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data/features_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 3.4  WordCloud par classe
# ============================================================

try:
    fig, axes = plt.subplots(1, len(df['label'].unique()), figsize=(16, 5))
    if len(df['label'].unique()) == 1:
        axes = [axes]

    for ax, label in zip(axes, df['label'].unique()):
        corpus = ' '.join(df[df['label'] == label]['text'].values)
        # Nettoyage minimal pour le nuage de mots
        corpus = re.sub(r'https?://\S+', '', corpus)
        corpus = re.sub(r'[^a-zA-Z\s]', '', corpus).lower()

        wc_color = {'ham': 'Greens', 'spam': 'Reds', 'phishing': 'Oranges'}.get(label, 'Blues')
        wc = WordCloud(
            width=400, height=300,
            background_color='white',
            colormap=wc_color,
            max_words=80
        ).generate(corpus)
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(f'Classe : {label.upper()}', fontweight='bold', fontsize=13)

    plt.suptitle('Mots les plus fréquents par classe', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('data/wordclouds.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'⚠  WordCloud ignoré : {e}')

## 4. Statistiques descriptives complètes

In [ ]:
# Statistiques détaillées par classe
stats = df.groupby('label')[['word_count', 'char_count', 'url_count', 'exclamation']].agg(
    ['mean', 'median', 'std', 'min', 'max']
).round(2)

print('📊 Statistiques descriptives par classe')
print('=' * 70)
print(stats.to_string())

print('\n📋 Résumé du dataset')
print(f'  Total emails       : {len(df)}')
print(f'  Classes présentes  : {list(df["label"].unique())}')
print(f'  Sources            : {list(df["source"].unique())}')
print(f'  Emails avec HTML   : {df["has_html"].sum()} ({df["has_html"].mean()*100:.1f}%)')
print(f'  Emails avec URLs   : {(df["url_count"]>0).sum()} ({(df["url_count"]>0).mean()*100:.1f}%)')

In [ ]:
# ============================================================
# 4.1  Corrélation entre features numériques
# ============================================================

num_features = ['word_count', 'char_count', 'url_count', 'exclamation', 'has_html']
corr_matrix  = df[num_features].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    square=True, linewidths=0.5
)
plt.title('Matrice de corrélation des features brutes', fontweight='bold')
plt.tight_layout()
plt.savefig('data/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Inspection d'exemples annotés

> Cette étape est cruciale : voir des exemples réels permet de mieux comprendre les patterns à détecter.

In [ ]:
# Affichage de quelques exemples par classe
for label in df['label'].unique():
    print(f'\n{'='*60}')
    print(f'CLASSE : {label.upper()}')
    print('='*60)
    examples = df[df['label'] == label]['text'].head(2).values
    for i, ex in enumerate(examples, 1):
        preview = ex[:300].replace('\n', ' ')
        print(f'\n  Exemple {i} :')
        print(f'  {preview}...' if len(ex) > 300 else f'  {preview}')

## 6. Export du dataset brut

In [ ]:
# Export vers CSV pour le Notebook 02
output_path = DATA_DIR / 'emails_raw.csv'
df.to_csv(output_path, index=False, encoding='utf-8')

print(f'✅ Dataset exporté : {output_path}')
print(f'   Taille : {output_path.stat().st_size / 1024:.1f} KB')
print(f'   Colonnes : {list(df.columns)}')
print(f'   Lignes : {len(df)}')

# Vérification rapide
df_check = pd.read_csv(output_path)
assert len(df_check) == len(df), 'Erreur : le fichier exporté est incomplet !'
print('\n✅ Vérification OK. Le Notebook 02 peut utiliser ce fichier.')

## 7. Résumé & points clés pour la suite

### Ce que nous avons appris dans ce notebook

| Observation | Impact sur la suite |
|-------------|---------------------|
| **Déséquilibre de classes** | Utiliser SMOTE ou class_weight dans le NB 02 |
| **URLs présentes dans phishing** | Feature importante à extraire (NB 02) |
| **Points d'exclamation dans spam** | Feature simple mais discriminante |
| **HTML présent dans certains emails** | Nécessite suppression des balises (NB 02) |

### Risques identifiés (cf. Cahier de charges)
- ⚠️ **Déséquilibre** : traité en NB 02 via SMOTE / class weights
- ⚠️ **Données synthétiques** : à enrichir avec des corpus réels (SpamAssassin, Enron)
- ⚠️ **Concept drift** : les patterns actuels peuvent évoluer — prévoir réentraînement (NB 06)

### Prochain notebook
**NB 02 — Prétraitement & Feature Engineering** : nettoyage du texte, TF-IDF, extraction d'URLs, gestion du déséquilibre.